In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [2]:
from datetime import date, datetime, timedelta
from pathlib import Path

import geopandas as gpd
import pandas as pd

from locallib.pandas import *
from locallib.picarrodb import *
from locallib.query import survey_query, query_segments_table

from geopackage import write_veho_geopackage
from msapi import send_email_via_outlook_api
from query import *
from utils import camel_to_snake, snake_to_camel


EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
# --- Config Area
Conn = EU1_Conn
customer_name = 'Northern Gas Networks'
short_name = customer_name.replace(' ','')
#recipients = ['dsoler@picarro.com','Theo@veho-solutions.co.uk']
recipients = ['dsoler@picarro.com']
send_email = True
#send_email = False

#process_date = datetime.now()
process_date = datetime(2026, 9, 20, 9, 0, 0)
delta_time = process_date - timedelta(hours=6)
processing_flag = False

#Cols
COLS =      ['ReportId', 'ReportName', 'ReportTitle', 'ReportDate', 'BoundaryName', 'BoundaryType', 'DistributionPipePercentCovered', 'Labels']


In [4]:
# Read the processed reports csv
try:
    reports_processed = pd.read_csv(f'{short_name}_reports_processed.csv')
except FileNotFoundError:
    # Create an empty DataFrame if the file does not exist
    reports_processed = pd.DataFrame(columns=COLS)

# --- Report Layer
reports = get_reports_veho(customer_name=customer_name).execute(Conn)[COLS]
reports = reports[pd.to_datetime(reports['ReportDate']) >= delta_time]
reports = reports[~reports['ReportId'].isin(reports_processed['ReportId'])]
if not reports.empty:
    reports_veho = reports.copy()  
    reports_veho.rename(columns={'DistributionPipePercentCovered':'DistmainsCoveragePct','BoundaryName':'BoundaryId'}, inplace=True)
    reports_veho.columns = [camel_to_snake(col) for col in reports_veho.columns]
    processing_flag = True
else:
    processing_flag = False


In [6]:
if processing_flag:
    # --- Survey Layer
    reports.db.set_query(survey_query(report_table='#TempReport'))
    surveys = reports.db.execute(Conn, source_col='ReportId', temp_table_name='#TempReport')
    surveys_veho = surveys[['SurveyId','ReportId','Tag','SurveyorUnit','AnalyzerSerialNumber','StartDateTimeSurvey','EndDateTimeSurvey','DurationMinutes']].copy()
    surveys_veho.rename(columns={'SurveyId':'SessionId','AnalyzerSerialNumber':'AnalyserSerial','StartDateTimeSurvey':'StartUtc','EndDateTimeSurvey':'EndUtc','DurationMinutes':'DurationMin','Tag':'SurveyTag'}, inplace=True)
    surveys_veho['StartUtc'] = pd.to_datetime(surveys_veho['StartUtc'])
    surveys_veho['EndUtc'] = pd.to_datetime(surveys_veho['EndUtc'])
    surveys_veho['DurationMin'] = surveys_veho['DurationMin'].astype(int)
    surveys_veho.columns = [camel_to_snake(col) for col in surveys_veho.columns]


    # --- Segment Layer

    surveys.db.set_query(query_segments_table(survey_table='#TempSurvey'))
    segments = surveys.db.execute(Conn, source_col='SurveyId', temp_table_name='#TempSurvey')
    segments_veho = pd.merge(
        surveys[['SurveyId','ReportId']],
        segments[['SurveyId','Id','Shape','StartEpoch','EndEpoch','DurationSeconds','LengthMeters','CarSpeedMedian']],
        on='SurveyId'
    )
    segments_veho.rename(
        columns={
            'SurveyId': 'SessionId',
            'Id': 'SegmentId',
            'Shape': 'Geometry',
            'DurationSeconds': 'DurationS',
            'LengthMeters': 'LengthM',
            'CarSpeedMedian': 'CarSpeedMed'
        },
        inplace=True
    )
    segments_veho.columns = [camel_to_snake(col) for col in segments_veho.columns]

    # --- FOV Layer
    gdf = gpd.GeoDataFrame(
        segments_veho,
        geometry=gpd.GeoSeries.from_wkt(segments_veho['geometry']),
        crs="EPSG:4326"
    )
    gdf = gdf.to_crs(epsg=27700)
    segments_veho = gdf

    query = f"""SELECT SurveyId as SurveyId, FieldOfView.STAsText() as FieldOfView FROM SurveyResult SR WHERE SR.SurveyId IN (SELECT SurveyId FROM #TempSurvey)"""
    surveys.db.set_query(query)
    fov = pd.merge(
        surveys[['SurveyId', 'ReportId']],
        surveys.db.execute(Conn, source_col='SurveyId', temp_table_name='#TempSurvey')[['SurveyId', 'FieldOfView']],
        on='SurveyId'
    )
    fov_veho = fov.copy()
    fov_veho.rename(columns={'SurveyId':'SessionId','FieldOfView':'geometry'}, inplace=True)
    fov_veho.columns = [camel_to_snake(col) for col in fov_veho.columns]

    # Convert geometry column to GeoSeries with EPSG:4326, then set CRS to 27700
    gdf_fov = gpd.GeoDataFrame(
        fov_veho,
        geometry=gpd.GeoSeries.from_wkt(fov_veho['geometry']),
        crs="EPSG:4326"
    )
    gdf_fov = gdf_fov.to_crs(epsg=27700)
    fov_veho = gdf_fov

    hyear = process_date.year
    output_gpkg = Path(f"output/{short_name}{str(hyear)[-2:]}_survey_{process_date.isoformat()}.gpkg")
    write_veho_geopackage(
        output_gpkg,
        reports_veho,
        surveys_veho,
        segments_veho,
        fov_veho,
    )
    print(f'GeoPackage written to {output_gpkg.resolve()}')

    # --- Save the processed reports csv

    reports_processed = pd.concat([reports_processed, reports], ignore_index=True)
    reports_processed.to_csv(f'{short_name}_reports_processed.csv', index=False)


GeoPackage written to /home/sandbox/personal-repos/Jira_Tickets/DA-3789/output/NorthernGasNetworks26_survey_2026-09-20T09:00:00.gpkg


In [ ]:
if processing_flag:
    # --- Send the email
    if send_email:
        for rcpt in recipients:
                try:
                    send_email_via_outlook_api(
                        subject=f'{customer_name} - GeoPackage {output_gpkg}',
                        body=f'Generated on {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}.',
                        recipient=rcpt,
                        attachments=[output_gpkg]
                    )
                    print(f"✅ Email sent successfully to {rcpt}")

                except Exception as e:
                    print(f"❌ Failed to send email to {rcpt}: {str(e)}")

✅ Email sent successfully to dsoler@picarro.com
